# Module 5 Code Exercise

In [ ]:
data_dir = "file_path"

count_method = 'n' # 'c' or 'n' # n = n tokens, c = distinct token (term) count
tf_method = 'sum' # sum, max, log, double_norm, raw, binary
tf_norm_k = .5 # only used for double_norm
idf_method = 'standard' # standard, max, smooth
gradient_cmap = 'YlGnBu' # YlGn, GnBu, YlGnBu; For tables; see https://matplotlib.org/3.1.0/tutorials/colors/colormaps.html 

OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
SENTS = OHCO[:4]
PARAS = OHCO[:3]
CHAPS = OHCO[:2]
BOOKS = OHCO[:1]

bag = CHAPS

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly_express as px

In [4]:
sns.set()
%matplotlib inline

In [5]:
LIB = pd.read_csv(data_dir + "LIB.csv").set_index(BOOKS)
TOKEN = pd.read_csv(data_dir + 'TOKEN.csv').set_index(OHCO)
VOCAB = pd.read_csv(data_dir + 'VOCAB.csv').set_index('term_id')
DOC = pd.read_csv(data_dir + "DOC.csv").set_index(PARAS)

In [6]:
VOCAB = VOCAB[~VOCAB.term_str.isna()]
TOKEN = TOKEN[~TOKEN.term_str.isna()]

In [7]:
TOKEN['term_id'] = TOKEN.term_str.map(VOCAB.reset_index().set_index('term_str').term_id)
VOCAB['pos_max'] = TOKEN.groupby(['term_id', 'pos']).pos.count().unstack().idxmax(1)

In [8]:
def get_tfidf(token_df, bag, count_method='n', tf_method='sum', idf_method='standard'):
    """
    Compute a TFIDF matrix from a TOKEN dataframe.

    Parameters
    ----------
    token_df     : pd.DataFrame — TOKEN table with OHCO multi-index and term_id column
    bag          : list        — OHCO levels defining the document unit, e.g. BOOKS or CHAPS
    count_method : str         — 'n' for raw token counts, 'c' for binary (term present/absent)
    tf_method    : str         — how to normalize TF: 'sum', 'max', 'log', 'raw', 'binary'
    idf_method   : str         — how to compute IDF: 'standard', 'max', 'smooth'

    Returns
    -------
    TFIDF : pd.DataFrame — matrix of shape (n_docs x n_terms)
    """

    BOW = (
        token_df
        .groupby(bag + ['term_id'])
        .term_id.count()
        .to_frame()
        .rename(columns={'term_id': 'n'})
    )
    BOW['c'] = BOW.n.astype('bool').astype('int')

    DTCM = BOW[count_method].unstack().fillna(0).astype('int')

    if tf_method == 'sum':
        TF = (DTCM.T / DTCM.T.sum()).T

    elif tf_method == 'max':
        TF = (DTCM.T / DTCM.T.max()).T

    elif tf_method == 'log':
        TF = np.log10(1 + DTCM)

    elif tf_method == 'raw':
        TF = DTCM

    elif tf_method == 'binary':
        TF = DTCM.astype('bool').astype('int')

    else:
        raise ValueError(f"Unknown tf_method: '{tf_method}'. "
                         f"Choose from: 'sum', 'max', 'log', 'raw', 'binary'")

    N  = DTCM.shape[0]
    DF = DTCM[DTCM > 0].count()

    if idf_method == 'standard':
        IDF = np.log10(N / DF)

    elif idf_method == 'max':
        IDF = np.log10(DF.max() / DF)

    elif idf_method == 'smooth':
        IDF = np.log10((1 + N) / (1 + DF)) + 1

    else:
        raise ValueError(f"Unknown idf_method: '{idf_method}'. "
                         f"Choose from: 'standard', 'max', 'smooth'")

    TFIDF = TF * IDF

    return TFIDF

# Question 1:

In [ ]:
TFIDF_books = get_tfidf(
    token_df     = TOKEN,
    bag          = BOOKS,
    count_method = 'n',         # raw token counts
    tf_method    = 'sum',       # normalize by total tokens per book
    idf_method   = 'standard'   # log(N / DF)
)

tfidf_book_sum = TFIDF_books.sum().to_frame('tfidf_sum')

top20_books = (
    tfidf_book_sum
    .join(VOCAB[['term_str', 'pos_max']])
    .sort_values('tfidf_sum', ascending=False)
    .head(20)
)

top20_books.style.background_gradient(cmap=gradient_cmap)

,tfidf_sum,term_str,pos_max
term_id,,,
26302,0.009838,pierre,NNP
11648,0.006763,elinor,NNP
19306,0.006504,israel,NNP
38673,0.005857,vernon,NNP
2644,0.005394,babbalanja,NNP
22176,0.004940,media,NNP
5540,0.004324,catherine,NNP
21823,0.004316,marianne,NNP
29073,0.004167,reginald,NNP


In [21]:
px.bar(
    top20_books.reset_index(),
    x='term_str', y='tfidf_sum', color='pos_max',
    title='Top 20 Terms by TFIDF Sum With Count Method as "n"',
    labels={'term_str': 'Term', 'tfidf_sum': 'TFIDF Sum', 'pos_max': 'POS'},
    height=450
)

The bar chart above shows the top 20 words in the corpus when using TF-IDF sum with the count method set to 'n'. 

# Question 2

In [14]:
TFIDF_chaps = get_tfidf(
    token_df     = TOKEN,
    bag          = CHAPS,
    count_method = 'n',
    tf_method    = 'sum',
    idf_method   = 'standard'
)

tfidf_chap_sum = TFIDF_chaps.sum().to_frame('tfidf_sum')

top20_chaps = (
    TFIDF_chaps.sum()
    .to_frame('tfidf_sum')
    .join(VOCAB[['term_str', 'pos_max']])
    .sort_values('tfidf_sum', ascending=False)
    .head(20)
)

top20_chaps.style.background_gradient(cmap=gradient_cmap, high=1)

,tfidf_sum,term_str,pos_max
term_id,,,
31648,1.349173,she,PRP
16730,1.331294,her,PRP$
26302,1.154340,pierre,NNP
40387,0.756695,you,PRP
23260,0.706684,mr,NNP
17566,0.664376,i,PRP
39540,0.594694,whale,NN
23261,0.593591,mrs,NNP
35574,0.586061,thou,NN


In [22]:
px.bar(
    top20_chaps.reset_index(),
    x='term_str', y='tfidf_sum', color='pos_max',
    title='Top 20 Terms by TFIDF Sum — Chapters as Bag with count method ="n"',
    labels={'term_str': 'Term', 'tfidf_sum': 'TFIDF Sum', 'pos_max': 'POS'},
    height=450
)

In [23]:
comparison = pd.DataFrame({
    'Books — Term':    top20_books['term_str'].values,
    'Books — POS':     top20_books['pos_max'].values,
    'Chapters — Term': top20_chaps['term_str'].values,
    'Chapters — POS':  top20_chaps['pos_max'].values,
}, index=range(1, 21))
comparison.index.name = 'Rank'

overlap = set(top20_books['term_str']) & set(top20_chaps['term_str'])

In [24]:
pos_books = top20_books['pos_max'].value_counts().rename('Books')
pos_chaps = top20_chaps['pos_max'].value_counts().rename('Chapters')
pd.concat([pos_books, pos_chaps], axis=1).fillna(0).astype(int)

,Books,Chapters
pos_max,,
NNP,20,9
PRP,0,4
PRP$,0,3
NN,0,3
JJ,0,1


When we use chapter as the bag, we see that the top 20 changes from all names, to now being a mixutre of pronouns, common nouns, and proper nouns. 